# Session 1: AI Overview, Use Cases & Development Tools
**Ask IRA — Investment Research Agent**

## Learning Objectives
1. Understand the AI landscape and key terminology (LLM, RAG, Agent, MCP)
2. Identify investment research use cases for AI
3. Set up the development environment for the course
4. Run the Ask IRA application end-to-end

---

## Part A — AI Landscape Overview

### Key Concepts

| Concept | Description | Role in Ask IRA |
|---------|-------------|-----------------|
| **LLM** | Large Language Model (Claude, GPT-4) | Core reasoning engine for analysis |
| **RAG** | Retrieval-Augmented Generation | Grounds answers in internal knowledge base |
| **Agent** | Autonomous LLM-driven decision maker | Orchestrates research workflow |
| **MCP** | Model Context Protocol | Standardized tool/server interface |
| **LangGraph** | Stateful graph framework | Manages agent state and flow |
| **Embedding** | Vector representation of text | Enables semantic search |

### AI-Native Engineering Principles
1. **Separation of concerns**: LLM handles reasoning; code handles data, state, and control flow
2. **Observability first**: Every LLM call traced, every agent decision logged
3. **Guardrails by default**: Input/output safety checks at every boundary
4. **Vendor independence**: MCP enables swapping models/services without code changes
5. **Iterative refinement**: Prompt-as-code with version control, A/B testing, eval suites

In [ ]:
# Part B — Verify Development Environment
import sys
print(f"Python version: {sys.version}")

required_packages = [
    "fastapi", "langchain", "langgraph", "chromadb",
    "sentence_transformers", "pydantic", "httpx",
]

missing = []
for pkg in required_packages:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"  [OK] {pkg}")
    except ImportError:
        missing.append(pkg)
        print(f"  [MISSING] {pkg}")

if missing:
    print(f"\nInstall missing packages: pip install {' '.join(missing)}")
else:
    print("\nAll required packages are installed!")

---
## Part C — Use Cases for AI in Investment Research

### What Ask IRA Can Do
- **Company Analysis**: Research financials, competitive position, growth drivers
- **Sentiment Analysis**: Aggregate news and social media sentiment for any ticker
- **Macro Research**: Analyze GDP, inflation, interest rates, employment trends
- **Risk Assessment**: Apply portfolio risk frameworks and position sizing rules
- **Report Generation**: Produce professional investment research reports

### Example Queries
- *"What is Apple's current outlook?"*
- *"Analyze MSFT stock considering macro conditions"*
- *"Should I invest in Google given the antitrust concerns?"*

In [ ]:
# Part D — Quick Test: Import the Ask IRA project
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Test that data loader works
from data.loader import load_all_seed_data

seed = load_all_seed_data()
for key, val in seed.items():
    if isinstance(val, list):
        print(f"  {key}: {len(val)} items")
    elif isinstance(val, dict):
        print(f"  {key}: {len(val)} keys")

print(f"\nTotal seed data collections: {len(seed)}")

In [ ]:
# Part E — Test MCP Server Discovery
# Verify all 4 MCP servers initialize and respond
import asyncio
from src.mcp_servers.registry import MCPRegistry
from src.mcp_servers.base import MCPRequest

async def test_servers():
    registry = MCPRegistry()
    print(f"Registered servers: {registry.server_names}\n")
    
    for name in registry.server_names:
        server = registry.get_server(name)
        resp = await server.handle(MCPRequest(query=f"What is AAPL outlook?"))
        print(f"[{name}]: {resp.content[:80]}...")
    
    print("\nAll MCP servers responding!")

await test_servers()

---
## Part F — Architecture Overview

```
User Query
    |
    v
[Input Guardrails]  -->  BLOCKED if PII/jailbreak detected
    |
    v
[Supervisor Agent]  -->  Plans research dimensions, selects MCP servers
    |
    v
[Researcher Agent]  -->  Dispatches MCP servers in parallel + RAG
    |  market_data     sentiment      macro      internal_kb
    v
[Analyst Agent]     -->  Synthesizes findings, rates confidence
    |
    v
[Writer Agent]      -->  Produces structured research report
    |
    v
[Output Guardrails] -->  Checks for hallucinations, sensitive content
    |
    v
Final Investment Research Report
```

### Key Design Decisions
- **Stateful DAG** via LangGraph ensures traceability and checkpointing
- **Parallel dispatch** via `asyncio.gather` for MCP servers
- **Hybrid RAG** (BM25 + dense) with cross-encoder re-ranking for relevant context
- **Guardrails at both ends**: input (PII, jailbreak) and output (hallucination, confidential data)

---
## Summary

In this session we:
1. Surveyed the AI landscape and key terminology
2. Verified the development environment
3. Identified investment research use cases
4. Imported and tested the Ask IRA codebase
5. Tested all 4 MCP servers
6. Reviewed the multi-agent architecture

**Next Session**: Python Fundamentals for AI (NumPy, Pandas, Data Analysis)